Part 1

Import and load datasets

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

DATA_DIR = Path("/data")
csv_files = sorted(DATA_DIR.rglob("*.csv"))

datasets = {}

for file in csv_files:
    name = file.stem
    df = pd.read_csv(file)
    datasets[name] = df
    print(f"{name} loaded")


In [ ]:
Get targets for each dataset

In [ ]:
drop_columns = {
    "breast cancer": ["id"],
    "zoo": ["animal_name", "animal"],
}

target_candidates = {
    "iris": ["species", "class", "target"],
    "wine": ["target", "class"],
    "breast cancer": ["diagnosis", "target", "class"],
    "optical digits": ["target", "class", "digit", "label"],
    "glass2": ["type", "type_of_glass", "class"],
    "chess": ["class", "target"],
    "zoo": ["type", "class", "target"],
}

for name, df in datasets.items():
    key = name.lower()
    candidates = next(
        (cols for keyword, cols in target_candidates.items()
         if keyword in key),
        []
    )

    matches = [
        col for col in df.columns
        if str(col).strip().lower() in candidates
    ]

    print(f"\n{name}")
    print("Columns:", df.columns.tolist())
    print("Likely target:", matches or "Needs manual inspection")

By using splits we can limit the sizes of the dataset to vary the training set size. 
We will first try a naive approach.
To do so, we split every dataset the same way with Stratified 10-Fold Cross-Validation with shuffling.

In [ ]:
cv_splits = {}
for name, df in datasets.items():
    X = df.drop(columns="target")
    y = df["target"]

    skf = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )
    cv_splits[name] = list(skf.split(X))

    print(name, X.shape, y.shape, y.nunique())

In [ ]:
We then fit a simple linear regression model and calculate the accuracy 

In [ ]:
cv_splits[name] = list(skf.split(X, y))

results = {}
for name, df in datasets.items():
    X = pd.get_dummies(df.drop(columns="target"))
    y = df["target"].reset_index(drop=True)

    fold_accuracies = []

    for train_idx, test_idx in cv_splits[name]:
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=2000)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        fold_accuracies.append(
            accuracy_score(y_test, y_pred)
        )

    results[name] = {
        "mean_accuracy": np.mean(fold_accuracies),
        "std_accuracy": np.std(fold_accuracies),
        "fold_accuracies": fold_accuracies,
    }

summary = pd.DataFrame(results).T[
    ["mean_accuracy", "std_accuracy"]
]

print(summary.sort_values("mean_accuracy", ascending=False))

This will serve as our baseline. 
We will now define the models that we will be testing with

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM": SVC(kernel="rbf"),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=1000,
        early_stopping=False,
        random_state=42
    ),
}

sample_fractions = [1.0, 0.75, 0.50, 0.25, 0.10]
n_seeds = 5

And we will create a function for classifying models with a pipeline

In [ ]:
def make_pipeline_for_model(df, classifier):
    X = df.drop(columns="target")

    numeric_cols = X.select_dtypes(include="number").columns.tolist()
    categorical_cols = X.select_dtypes(
        exclude="number"
    ).columns.tolist()

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(("num", numeric_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(("cat", categorical_pipe, categorical_cols))

    preprocess = ColumnTransformer(transformers)

    return Pipeline([
        ("preprocess", preprocess),
        ("classifier", clone(classifier)),
    ])

We will now run with differnt sizes

In [ ]:
results = []

for name, df in datasets.items():
    X = df.drop(columns="target").reset_index(drop=True)
    y = df["target"].reset_index(drop=True)

    for fold, (train_idx, test_idx) in enumerate(
        cv_splits[name]
    ):
        X_pool = X.iloc[train_idx]
        y_pool = y.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_test = y.iloc[test_idx]

        n_classes = y_pool.nunique()

        for fraction in sample_fractions:
            n_samples = max(
                n_classes,
                int(round(len(train_idx) * fraction))
            )

            # Do not request more observations than available
            n_samples = min(n_samples, len(train_idx))

            for seed in range(n_seeds):
                try:
                    if n_samples == len(train_idx):
                        selected = np.arange(len(train_idx))
                    else:
                        selected, _ = train_test_split(
                            np.arange(len(train_idx)),
                            train_size=n_samples,
                            stratify=y_pool,
                            random_state=seed
                        )

                    X_sub = X_pool.iloc[selected]
                    y_sub = y_pool.iloc[selected]

                    # Some classes may be absent in a tiny subset.
                    if y_sub.nunique() < 2:
                        continue

                    for model_name, classifier in models.items():
                        model = make_pipeline_for_model(
                            df, classifier
                        )
                        model.fit(X_sub, y_sub)
                        pred = model.predict(X_test)

                        results.append({
                            "dataset": name,
                            "fold": fold,
                            "seed": seed,
                            "model": model_name,
                            "fraction": fraction,
                            "n_train": len(selected),
                            "accuracy": accuracy_score(y_test, pred),
                        })

                except ValueError as e:
                    # Tiny or rare-class subsets may be infeasible.
                    print(
                        f"Skipping {name}, fold={fold}, "
                        f"fraction={fraction}, seed={seed}: {e}"
                    )

results_df = pd.DataFrame(results)

In [ ]:
We will now summarise and visualise it for analysis

In [ ]:
summary = (
    results_df
    .groupby(["dataset", "model", "fraction"])
    .agg(
        mean_accuracy=("accuracy", "mean"),
        std_accuracy=("accuracy", "std"),
        runs=("accuracy", "count"),
        mean_train_size=("n_train", "mean"),
    )
    .reset_index()
)

summary.head()



for dataset in summary["dataset"].unique():
    subset = summary[summary["dataset"] == dataset]

    fig, ax = plt.subplots(figsize=(8, 5))

    for model_name, group in subset.groupby("model"):
        group = group.sort_values("mean_train_size")

        ax.plot(
            group["mean_train_size"],
            group["mean_accuracy"],
            marker="o",
            label=model_name,
        )

        ax.fill_between(
            group["mean_train_size"],
            group["mean_accuracy"] - group["std_accuracy"],
            group["mean_accuracy"] + group["std_accuracy"],
            alpha=0.12,
        )

    ax.set(
        title=f"Learning curves — {dataset}",
        xlabel="Number of training examples",
        ylabel="Mean accuracy",
    )
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

Report: The Effect of Limited Training Data on Classification
1. Introduction

Machine learning models require sufficient training data to learn patterns that generalise to unseen observations. When training data are limited, predictive performance may decrease and results may become more sensitive to the particular observations selected. However, the effect depends on the learning algorithm, dataset characteristics and model complexity.

This investigation examines how reducing training data affects five classifiers: Logistic Regression, K-Nearest Neighbours (KNN), Decision Tree, Support Vector Machine (SVM) and Multilayer Perceptron (MLP). Experiments use seven datasets: Iris, Wine, Breast Cancer Wisconsin, Optical Digits, Glass2, Chess KR-vs-KP and Zoo.

The study investigates how performance changes with sample size, which classifiers are most robust, whether overfitting becomes more pronounced, and how reliably classifiers can be compared on small datasets.

2. Experimental Design
2.1 Datasets and classifiers

The datasets differ in sample size, dimensionality, class balance and feature type. This allows the investigation to consider whether the effects of data scarcity are consistent across different classification problems.

The five classifiers represent different learning approaches. Logistic Regression provides a relatively simple linear baseline. KNN relies on neighbouring observations and may be sensitive to individual examples. Decision Trees can learn complex decision boundaries but may overfit. SVMs use margin-based classification and regularisation, while MLPs can learn nonlinear relationships but introduce additional complexity through their architecture and training process.

2.2 Evaluation protocol

Stratified 10-fold cross-validation is used where class counts permit it. Each fold acts as a test set once, with the remaining observations forming the training pool. Within each training pool, subsets containing 100%, 75%, 50%, 25% and 10% of the available training observations are evaluated.

For each fold and sample size, all classifiers use the same selected training observations and test set. Repeated subsampling investigates how results vary with the choice of training examples. Small or imbalanced datasets require particular care because some classes may have too few observations for reliable 10-fold evaluation.

Preprocessing is fitted within each training subset to prevent information leakage. Numerical features are standardised where appropriate, while categorical features are encoded. Accuracy is the primary metric, supplemented by balanced accuracy or macro-F1 where class imbalance makes overall accuracy misleading.

Mean performance and standard deviation are reported for each configuration. Learning curves show how performance changes with training-set size, while training-versus-test performance helps identify overfitting.

3. Results and Discussion
3.1 Effect of reducing training data

The learning curves show how each classifier responds to progressively smaller training sets. The observed results indicate that [describe the overall trend and identify any exceptions].

Performance may decline because smaller samples provide less information about class boundaries and feature relationships. However, the rate of deterioration may differ between algorithms. A relatively stable learning curve suggests that a classifier can learn useful patterns from fewer observations, whereas a sharp decline suggests greater sensitivity to sample size.

Differences between datasets are also important. A sample size sufficient for a simple, low-dimensional problem may be inadequate for a more complex dataset. Results should therefore be interpreted using both the number of training observations and the characteristics of each dataset.

3.2 Classifier robustness

Robustness depends on both mean predictive performance and variability across repeated runs. The results show that [identify the most consistent classifiers] were comparatively stable, while [identify the most variable classifiers] showed greater fluctuations.

These differences may reflect the learning mechanisms involved. Logistic Regression has a relatively constrained decision boundary, which can help limit variance but may cause underfitting. KNN depends on the available local examples, making its predictions potentially sensitive to subsampling. Decision Trees may change substantially when a small number of observations are removed.

SVM performance depends on regularisation and kernel choice. The MLP may be sensitive to training-set size and random initialisation. These explanations should be considered alongside the observed results rather than assumed to apply equally to every dataset.

3.3 Overfitting and model comparison

Overfitting is investigated by comparing training and test performance. A large gap indicates that a model fits its training examples more successfully than unseen observations. If this gap increases as training data decrease, it suggests that generalisation is becoming more difficult.

The results show [describe whether a widening gap occurs and for which classifiers]. Flexible models such as Decision Trees and MLPs are useful cases to examine, although any classifier can generalise poorly under unsuitable settings.

Model rankings must also be interpreted in relation to variability. If the difference between two mean accuracy scores is small compared with fluctuations across folds, the evidence for a genuine performance advantage is weak. Selecting hyperparameters using the same test results can further bias comparisons, so tuning must be restricted to training data.

3.4 Reliability on very small datasets

Small test sets produce uncertain performance estimates because individual predictions have a relatively large influence on the final score. This is particularly important for Zoo and Glass2 if some classes contain very few examples.

The observed standard deviations and learning curves indicate [describe the level of variability and whether model rankings change across runs]. Large variability suggests that conclusions based on one split would be unreliable. If rankings change across folds or training subsets, the evidence may not support identifying one consistently superior classifier.

Repeated cross-validation helps investigate this uncertainty but does not create additional independent observations. Standard deviations describe observed variability; they should not automatically be interpreted as confidence intervals or formal evidence of statistical significance.

3.5 Implications for experimental design

The findings demonstrate the importance of consistent evaluation. Using identical folds and training subsets makes classifier comparisons fairer, while training-only preprocessing prevents leakage. Repeated sampling reveals variability that a single train/test split would conceal.

The results also highlight the need to control model complexity and document hyperparameter choices. Stratification helps preserve class proportions but cannot overcome a shortage of minority-class examples. Alternative validation strategies may produce different estimates, particularly for very small datasets.

Consequently, model selection should consider predictive performance, variability and the limitations of the evaluation procedure rather than relying exclusively on mean accuracy.

4. Limitations

The study covers seven datasets and five baseline classifiers, so its findings may not generalise to all machine learning problems. Results also depend on model configurations and hyperparameters; alternative settings could produce different learning curves.

Small class counts limit the feasibility of some cross-validation configurations, and repeated folds are not independent experiments. Furthermore, differences in accuracy may be difficult to interpret when test sets are small or classes are imbalanced. These limitations mean that apparent differences between classifiers should be interpreted cautiously.

5. Conclusion

This investigation examines how training-set size affects classification performance, stability and generalisation. By comparing five different learning algorithms across seven datasets, it considers the interaction between data scarcity, model complexity and dataset characteristics.

The experimental results suggest that [summarise the main observed trend], while [identify the key difference between classifiers]. Variability across repeated runs demonstrates [summarise the evidence about reliability].

Overall, the study shows that selecting a classifier for small-data problems requires more than identifying the highest mean accuracy. Learning curves, variability, training-versus-test performance and a leakage-free evaluation protocol are essential for drawing defensible conclusions from limited data.